# 加载RubricHub

In [1]:
!pip install --upgrade jupyter ipywidgets

In [1]:
import os

# 🔥 使用国内镜像站 - 最稳定的方案（不需要 VPN）
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# 清除代理设置，避免冲突
if 'http_proxy' in os.environ:
    del os.environ['http_proxy']
if 'https_proxy' in os.environ:
    del os.environ['https_proxy']

print("✅ 已配置使用 HuggingFace 国内镜像站")
print("镜像地址: https://hf-mirror.com")

✅ 已配置使用 HuggingFace 国内镜像站
镜像地址: https://hf-mirror.com


In [3]:
import os
from datasets import load_dataset

# 保留镜像
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

print("🔄 以流式模式加载（绕过本地arrow cast）...")
ds = load_dataset("sojuL/RubricHub_v1", streaming=True)

# 查看可用 split
print(ds)

# 取几条样本验证
train_iter = iter(ds["train"])
for i in range(3):
    x = next(train_iter)
    print(f"sample {i+1} keys:", list(x.keys())[:8], "...")

🔄 以流式模式加载（绕过本地arrow cast）...
IterableDatasetDict({
    train: IterableDataset({
        features: ['prompt', 'data_source', 'ability', 'reward_model', 'extra_info', 'Rubrics', '__index_level_0__'],
        num_shards: 7
    })
})
sample 1 keys: ['prompt', 'data_source', 'ability', 'reward_model', 'extra_info', 'Rubrics', '__index_level_0__'] ...
sample 2 keys: ['prompt', 'data_source', 'ability', 'reward_model', 'extra_info', 'Rubrics', '__index_level_0__'] ...
sample 3 keys: ['prompt', 'data_source', 'ability', 'reward_model', 'extra_info', 'Rubrics', '__index_level_0__'] ...


In [4]:
import os
import shutil
from pathlib import Path

target_dir = Path(r"D:\学校学习\李侃老师科研\Evaluation\LLM-as-a-Judge鲁棒性分析\RobutsEval\RubricHub\RubricHub_v1")
target_dir.mkdir(parents=True, exist_ok=True)

# HuggingFace 缓存根目录
hf_home = Path(os.getenv("HF_HOME", Path.home() / ".cache" / "huggingface"))
repo_cache = hf_home / "hub" / "datasets--sojuL--RubricHub_v1"
snapshots_dir = repo_cache / "snapshots"

if not snapshots_dir.exists():
    raise FileNotFoundError(f"未找到缓存目录: {snapshots_dir}")

# 选最新 snapshot
snapshot_list = [p for p in snapshots_dir.iterdir() if p.is_dir()]
if not snapshot_list:
    raise FileNotFoundError(f"未找到任何 snapshot: {snapshots_dir}")

src = max(snapshot_list, key=lambda p: p.stat().st_mtime)
print("源目录:", src)
print("目标目录:", target_dir)

# 复制（可覆盖已存在文件）
for item in src.iterdir():
    dst = target_dir / item.name
    if item.is_dir():
        shutil.copytree(item, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(item, dst)

print("✅ 复制完成")

源目录: C:\Users\Zhouxj's Lenovo\.cache\huggingface\hub\datasets--sojuL--RubricHub_v1\snapshots\3837d55971473a872e84879c88f708b8da3ec2ef
目标目录: D:\学校学习\李侃老师科研\Evaluation\LLM-as-a-Judge鲁棒性分析\RobutsEval\RubricHub\RubricHub_v1
✅ 复制完成


已清理代理，开始通过国内镜像下载数据集...


LocalEntryNotFoundError: An error happened while trying to locate the files on the Hub and we cannot find the appropriate snapshot folder for the specified revision on the local disk. Please check your internet connection and try again.

In [ ]:
# 2. 将数据集保存到你指定的文件夹路径
# 请将下面的路径替换为你想要保存的实际路径，例如 "D:/datasets/RubricHub_v1" 或 "./my_dataset"
ds.save_to_disk("D:\\学校学习\\李侃老师科研\\Evaluation\\LLM-as-a-Judge鲁棒性分析\\RobutsEval\\RubricHub\\RubricHub_v1")

## 加载原文的grader prompt

In [3]:
import json

with open('Prompt.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
    grader_template = data.get('grader_template')

print(grader_template)

Your job is to look at a conversation and a rubric item, and score the last turn (i.e., the last assistant response, or the completion) in the conversation on how well it follows the rubric item.

# Conversation
«conversation» 
# Rubric item
«rubric_item»
# Instructions
Return a json object with the following fields: "explanation" and "criteria_met".
- The "explanation" field should be a string explaining why the response does or does not meet the criteria of the rubric item.
- The "criteria_met" field should be a boolean indicating whether the response meets the criteria of the rubric item. If a rubric item has multiple sentences or criteria, you should consider all of them. If any of the criteria is not met, the answer should be false. Only return true is all of the criteria are met.
- One important exception to the above bullet point is that if a criteria says "such as", "for example", or "including", the response does not have to include all of the examples listed to meet the crite